# Linux 趣味工具设计与实现报告：字符错位“恶作剧”终端
## 一、作业任务回顾
本次作业要求：选择一个Linux工具，依据其功能设计一个有趣的场景，并完成设计、实现和效果展示。

本项目基于 **Bash Shell** 工具，利用Shell脚本的字符处理与命令执行能力，设计了一个“随机字符错位”的趣味终端场景——用户输入的命令中，约1/3的字母会被自动替换为QWERTY键盘上的下一个字符，打造一个“打字总是手滑”的搞怪终端效果。

---
## 二、设计思路
### 1. 核心功能目标
模拟“打字时不小心按到相邻键”的效果，对用户输入的命令行进行随机字符替换，再执行被修改后的命令，实现以下体验：
- 用户输入正常命令（如 `ls`），终端会随机将部分字符替换为键盘相邻键（如变成 `lt`、`ks` 等）
- 替换规则严格遵循QWERTY键盘的三行字母布局（q→w、w→e、a→s、z→x 等），大小写均支持
- 替换概率固定为1/3，既保证足够的“错位感”，又不会让命令完全无法识别
- 保持终端的基础交互逻辑，用户可正常输入、执行命令，输入 `exit` 退出该模式

### 2. 技术选型与原理
- **核心工具**：Bash Shell 脚本
- **关键技术点**：
  1.  **关联数组映射**：通过 `declare -gA TRANSFORM_MAP` 定义QWERTY键盘字母的错位映射表，覆盖大小写字母
  2.  **随机数控制**：利用 `RANDOM % 3` 实现1/3的随机替换概率
  3.  **字符级处理**：逐字符读取用户输入的命令行，对每个字母进行替换判断
  4.  **命令执行**：通过 `eval` 执行替换后的命令，并回显实际执行的内容，让用户直观看到错位效果

---
## 三、完整实现代码
```bash
#!/bin/bash

# Simple character transformer for terminal input
# Transforms characters with ~1/3 probability using QWERTY shift mapping

create_mapping() {
    # Define character mappings based on QWERTY keyboard rows
    declare -gA TRANSFORM_MAP
    
    # Row 1: QWERTYUIOP -> WERTYUIOPQ
    for i in {0..9}; do
        chars=("q" "w" "e" "r" "t" "y" "u" "i" "o" "p")
        next_chars=("w" "e" "r" "t" "y" "u" "i" "o" "p" "q")
        TRANSFORM_MAP["${chars[$i]}"]="${next_chars[$i]}"
        TRANSFORM_MAP["${chars[$i]^}"]="${next_chars[$i]^}"  # uppercase
    done
    
    # Row 2: ASDFGHJKL -> SDFGHJKLA
    for i in {0..8}; do
        chars=("a" "s" "d" "f" "g" "h" "j" "k" "l")
        next_chars=("s" "d" "f" "g" "h" "j" "k" "l" "a")
        TRANSFORM_MAP["${chars[$i]}"]="${next_chars[$i]}"
        TRANSFORM_MAP["${chars[$i]^}"]="${next_chars[$i]^}"  # uppercase
    done
    
    # Row 3: ZXCVBNM -> XCVBNMZ
    for i in {0..6}; do
        chars=("z" "x" "c" "v" "b" "n" "m")
        next_chars=("x" "c" "v" "b" "n" "m" "z")
        TRANSFORM_MAP["${chars[$i]}"]="${next_chars[$i]}"
        TRANSFORM_MAP["${chars[$i]^}"]="${next_chars[$i]^}"  # uppercase
    done
}

transform_char() {
    local char="$1"
    # Only transform alphabetical characters
    if [[ "$char" =~ [a-zA-Z] ]] && [[ -n "${TRANSFORM_MAP[$char]}" ]]; then
        # Randomly decide whether to transform (1 out of 3 chance)
        if (( RANDOM % 3 == 0 )); then
            echo -n "${TRANSFORM_MAP[$char]}"
            return
        fi
    fi
    # If not transforming, return original character
    echo -n "$char"
}

# Main function to process input
process_input() {
    echo "Character Transformer Active - ~1/3 transformation rate"
    echo "Starting new shell with character transformation..."
    
    # For demonstration, we'll create a simple readline-like function
    while true; do
        # Read line by line
        read -e -p "$ " input_line
        
        # Process each character in the line with transformation
        output=""
        for (( i=0; i<${#input_line}; i++ )); do
            char="${input_line:$i:1}"
            transformed=$(transform_char "$char")
            output="${output}${transformed}"
        done
        
        # Execute the transformed command
        echo "Executing: $output"
        eval "$output"
        
        # Break if user types exit
        [[ "$input_line" == "exit" ]] && break
    done
}

# Initialize mapping
create_mapping

# Start the transformed shell
process_input
```

---
## 四、安装与使用步骤
### 1. 安装脚本
在终端中执行以下命令，完成脚本创建、权限配置与别名设置：
```bash
# 创建脚本文件
cat > ~/simple_char_transform.sh << 'EOF'
# 复制上面的完整代码到这里（与前文一致）
EOF

# 赋予执行权限
chmod +x ~/simple_char_transform.sh

# 添加别名到bashrc，方便快速启动
echo 'alias char_transform="~/simple_char_transform.sh"' >> ~/.bashrc

# 刷新配置，使别名立即生效
source ~/.bashrc
```

### 2. 启动工具
在终端中输入以下命令，即可进入“字符错位模式”：
```bash
# 方式1：直接执行脚本
~/simple_char_transform.sh

# 方式2：使用别名启动（配置完成后）
char_transform
```

### 3. 退出工具
在错位终端中输入 `exit` 即可退出，恢复正常终端模式。

---
## 五、效果展示
### 1. 典型使用场景演示
| 用户输入命令 | 实际执行的命令（随机示例） | 效果说明 |
|--------------|----------------------------|----------|
| `ls`         | `lt` / `ks` / `ls`         | 列表命令随机错位，部分执行失败，部分正常运行 |
| `echo hello` | `echo hfllo` / `ecno hello` | 输出文本随机字符错位，出现“typo”效果 |
| `pwd`        | `qwd` / `pwe` / `pwd`      | 路径查询命令随机变化，部分报错 |
| `cd Documents` | `cd Documetns` / `ce Documents` | 目录切换命令随机错位，部分无法进入目标目录 |

### 2. 交互示例输出
```
Character Transformer Active - ~1/3 transformation rate
Starting new shell with character transformation...
$ ls
Executing: lt
bash: lt: command not found
$ ls
Executing: ls
Desktop  Documents  Downloads  Music  Pictures  Public  Templates  Videos
$ echo hello world
Executing: echo hfllo xpsme
hfllo xpsme
$ exit
```
![截图](image.png)
---
## 六、拓展与优化说明
### 1. 现有功能的局限性
- 仅支持Bash Shell，不兼容Zsh等其他Shell环境
- 无法处理特殊字符、管道符、重定向符的错位（仅针对字母生效）
- `read -e` 不支持复杂的终端编辑操作（如Ctrl+R历史搜索、方向键补全）

### 2. 可拓展方向
- 增加更多错位模式：如反向映射（下一个→上一个）、随机乱序、大小写翻转
- 支持自定义替换概率，用户可通过参数调整错位频率
- 扩展字符集，支持数字、符号的错位映射（如`1→2`、`!→@`）
- 实现更底层的终端输入劫持，让错位效果覆盖所有终端操作（而非仅脚本内的read交互）

---
## 七、总结
本项目基于Linux的Bash Shell工具，通过字符映射、随机替换和命令执行的组合，实现了一个充满趣味的“打字错位”终端场景。既利用了Shell脚本的基础功能，又通过创意场景赋予了工具新的玩法，同时也直观展示了Linux命令行交互的灵活性与可定制性。
